# Energy and enstrophy spectra
Plot instantaneous frames, several frames, or a pointwise frame average from `spectra.csv`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'ns2d_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from ns2d_plotting import (available_frames, curves_for_frames, positive_xy,
    read_csv, repository_root, save_figure, select_frames, use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()
SPECTRA_FILE = ROOT / 'output/spectra.csv'
ENERGY_FIGURE = ROOT / 'figures/energy_spectrum.pdf'
ENSTROPHY_FIGURE = ROOT / 'figures/enstrophy_spectrum.pdf'

# 'single': one curve and exactly one selected frame.
# 'multiple': one curve for every selected frame.
# 'average': one pointwise average over all selected frames.
MODE = 'single'

# Negative frame indices count from the end. Set to None to use the range.
FRAMES = [-1]
FRAME_START = None
FRAME_STOP = None
FRAME_STRIDE = 1

# True uses the running segment-mean columns instead of instantaneous data.
USE_SEGMENT_MEAN = False
LOG_SCALE = True
USE_TEX = True
FONT_SIZE = 16

In [ ]:
use_plot_style(USE_TEX, FONT_SIZE)
table = read_csv(SPECTRA_FILE)
frames = select_frames(available_frames(table), FRAMES, start=FRAME_START,
                       stop=FRAME_STOP, stride=FRAME_STRIDE)
energy_column = ('segment_mean_energy_spectrum' if USE_SEGMENT_MEAN
                 else 'energy_spectrum')
enstrophy_column = ('segment_mean_enstrophy_spectrum' if USE_SEGMENT_MEAN
                    else 'enstrophy_spectrum')
k, curves = curves_for_frames(table, frames,
                               [energy_column, enstrophy_column], MODE)
print(f'Selected frames: {frames}')

## Energy spectrum

In [ ]:
fig, axis = plt.subplots(figsize=(7.0, 5.2))
for curve in curves:
    x, y = k, np.asarray(curve[energy_column])
    if LOG_SCALE:
        x, y = positive_xy(x, y)
    axis.plot(x, y, label=curve['label'])
axis.set_xlabel(r'$k$')
axis.set_ylabel(r'$E(k)$')
if LOG_SCALE:
    axis.set_xscale('log')
    axis.set_yscale('log')
axis.grid(True, which='both', alpha=0.2)
axis.legend()
saved = save_figure(fig, ENERGY_FIGURE)
print(f'Wrote {saved}')
plt.show()

## Enstrophy spectrum

In [ ]:
fig, axis = plt.subplots(figsize=(7.0, 5.2))
for curve in curves:
    x, y = k, np.asarray(curve[enstrophy_column])
    if LOG_SCALE:
        x, y = positive_xy(x, y)
    axis.plot(x, y, label=curve['label'])
axis.set_xlabel(r'$k$')
axis.set_ylabel(r'$Z(k)$')
if LOG_SCALE:
    axis.set_xscale('log')
    axis.set_yscale('log')
axis.grid(True, which='both', alpha=0.2)
axis.legend()
saved = save_figure(fig, ENSTROPHY_FIGURE)
print(f'Wrote {saved}')
plt.show()